# TypiClust — Modified & Comparison Experiments

This notebook builds on `tpcrp_original.ipynb` and explores modifications to TypiClust
as well as comparisons against baseline active learning strategies.

**Prerequisites:** run `tpcrp_original.ipynb` first to generate:
- `results/train_features.npy` / `results/test_features.npy`
- `results/typiclust_original_history.json`

## Experiments
1. **Random baseline** — uniform random sampling
2. **Uncertainty sampling** — least-confidence (lowest max softmax)
3. **Margin sampling** — smallest top-1 minus top-2 softmax gap
4. **TypiClust ablation** — vary `max_clusters` ∈ {100, 500, 1000}
5. **Comparison plot** — all methods on one figure

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.utils import set_seed, load_results, save_results, plot_comparison, log
from src.active_learning import (
    TypiClust, RandomSelection, UncertaintySelection, MarginSelection,
    run_active_learning_loop,
)
from src.classifier import train_linear_probe, evaluate_linear_probe

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'Device: {DEVICE}')

## 1. Load Pre-computed Features

In [ ]:
train_feats  = np.load('../results/train_features.npy')
train_labels = np.load('../results/train_labels.npy')
test_feats   = np.load('../results/test_features.npy')
test_labels  = np.load('../results/test_labels.npy')

print(f'Train: {train_feats.shape}  |  Test: {test_feats.shape}')

## 2. Shared Configuration & Helpers

In [ ]:
INITIAL_BUDGET = 10
QUERY_BUDGET   = 10
N_ROUNDS       = 20
PROBE_EPOCHS   = 100
PROBE_LR       = 1e-3

def train_fn(labeled_idx, labels):
    return train_linear_probe(
        train_features = train_feats[labeled_idx],
        train_labels   = labels,
        epochs         = PROBE_EPOCHS,
        lr             = PROBE_LR,
        device         = DEVICE,
    )

def eval_fn(head):
    return evaluate_linear_probe(head, test_feats, test_labels, device=DEVICE)

def run(selector, name, on_model_update=None):
    log(f'Running {name}...')
    hist = run_active_learning_loop(
        features         = train_feats,
        labels           = train_labels,
        selector         = selector,
        initial_budget   = INITIAL_BUDGET,
        query_budget     = QUERY_BUDGET,
        n_rounds         = N_ROUNDS,
        train_fn         = train_fn,
        eval_fn          = eval_fn,
        on_model_update  = on_model_update,
        seed             = SEED,
    )
    save_results(hist, f'../results/{name.replace(" ", "_")}_history.json')
    return hist

## 3. Random Baseline

In [ ]:
hist_random = run(RandomSelection(train_feats, seed=SEED), 'Random')

## 4. Uncertainty Sampling

Queries unlabeled points with the lowest maximum softmax probability —  
i.e., where the current linear probe is least confident.

The `predict_proba_fn` is refreshed after each training round via `on_model_update`.

In [ ]:
def make_proba_fn(head):
    """Return a function that computes softmax probabilities for given pool indices."""
    head = head.to(DEVICE).eval()
    def _proba(indices):
        X = torch.tensor(train_feats[indices], dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(head(X), dim=1).cpu().numpy()
        return probs
    return _proba

# Placeholder until the first model is trained (round 1 uses random selection
# for the initial budget, then the model is updated before the first query).
unc_selector = UncertaintySelection(
    features         = train_feats,
    predict_proba_fn = lambda idx: np.ones((len(idx), 10)) / 10,  # uniform prior
)

def update_unc(model, labeled_idx):
    unc_selector.predict_proba_fn = make_proba_fn(model)

hist_uncertainty = run(unc_selector, 'Uncertainty', on_model_update=update_unc)

## 5. Margin Sampling

In [ ]:
margin_selector = MarginSelection(
    features         = train_feats,
    predict_proba_fn = lambda idx: np.ones((len(idx), 10)) / 10,
)

def update_margin(model, labeled_idx):
    margin_selector.predict_proba_fn = make_proba_fn(model)

hist_margin = run(margin_selector, 'Margin', on_model_update=update_margin)

## 6. TypiClust Ablation — `max_clusters`

The number of clusters K = min(|labeled| + B, max_clusters) controls the  
granularity of the coverage criterion.

In [ ]:
hist_typiclust_variants = {}
for mc in [100, 500, 1000]:
    sel = TypiClust(train_feats, max_clusters=mc, seed=SEED)
    hist_typiclust_variants[f'TypiClust_K{mc}'] = run(sel, f'TypiClust_K{mc}')

## 7. Comparison Plot

In [ ]:
original = load_results('../results/typiclust_original_history.json')

all_results = {
    'TypiClust (K_max=500)': original,
    'Random':                hist_random,
    'Uncertainty':           hist_uncertainty,
    'Margin':                hist_margin,
    **hist_typiclust_variants,
}

plot_comparison(all_results, save_path='../plots/typiclust_comparison.png')

## 8. Summary Table

In [ ]:
print(f'{'Method':<25}  {'Final acc':>10}  {'@ labeled':>10}')
print('-' * 50)
for name, hist in all_results.items():
    acc = hist['accuracies'][-1] * 100
    n   = hist['labelled_counts'][-1]
    print(f'{name:<25}  {acc:>9.2f}%  {n:>10d}')

## 9. Discussion

Fill in your analysis after running the experiments:

- **TypiClust vs. Random**: At what label budget does TypiClust start to clearly  
  outperform random sampling? Why does the gap widen or narrow at higher budgets?

- **TypiClust vs. Uncertainty/Margin**: Uncertainty methods require a trained model;  
  are they more or less effective than TypiClust in the very-low-budget regime?

- **Effect of max_clusters**: How sensitive is TypiClust to the K_max hyperparameter?
  What happens when K_max is too small (100) or very large (1000)?

- **Limitations**: What are failure modes of the typicality-based approach  
  (e.g., class imbalance within clusters, outlier sensitivity)?